# Deep Learning Based Contactless Palm Print Recognition
Based on the research paper by Shubh Agarwal, Anurag Parashar, Yash Gupta,
Vaibhav Vishwakarma, and Gurpreet Kour Khalsa — SRM Institute of Science & Technology.

**Dataset Structure:**
```
dataset/
    session1/
        00001.tiff   <- Person 1, session 1
        00002.tiff   <- Person 2, session 1
        ...
    session2/
        00001.tiff   <- Person 1, session 2
        00002.tiff   <- Person 2, session 2
        ...
```
- Each filename (e.g. 00001) = one unique person/class.
- session1 images are used for **TRAINING**.
- session2 images are used for **TESTING** (standard protocol for palmprint datasets).

## 0. Imports & Configuration

In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import regularizers


In [2]:
# CONFIGURATION  <- Edit DATASET_DIR to point to your archive folder
DATASET_DIR   = "dataset"          # folder containing session1 and session2
SESSION_TRAIN = "session1"         # folder used for training
SESSION_TEST  = "session2"         # folder used for testing
IMG_SIZE      = (128, 128)         # resize target (H, W)
BATCH_SIZE    = 32
EPOCHS        = 80
LEARNING_RATE = 1e-4
VAL_SPLIT     = 0.20               # fraction of training data used for validation
RANDOM_SEED   = 42
MODEL_SAVE    = "palmprint_model.h5"
OUTPUT_DIR    = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[INFO] GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"[WARNING] Could not set GPU memory growth: {e}")
else:
    print(f"[INFO] No GPU detected, running on CPU")

print(f"\n[CONFIG] Dataset configuration:")
print(f"  Dataset dir: {DATASET_DIR}")
print(f"  Training: {SESSION_TRAIN}")
print(f"  Testing: {SESSION_TEST}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

[INFO] No GPU detected, running on CPU

[CONFIG] Dataset configuration:
  Dataset dir: dataset
  Training: session1
  Testing: session2
  Image size: (128, 128)
  Batch size: 32
  Epochs: 80
  Learning rate: 0.0001


## 1. ROI Extraction
Automated ROI extraction optimised for black-background palmprint images.

In [3]:
def extract_roi(image_bgr):
    """
    Automated ROI extraction optimised for black-background palmprint images:
      1. Grayscale + simple threshold (black bg makes this trivial)
      2. Morphological cleanup
      3. Find largest contour (the palm)
      4. Crop bounding box with padding
    Falls back to full image if detection fails.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY) \
           if len(image_bgr.shape) == 3 else image_bgr.copy()

    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = 15
        x = max(0, x - pad)
        y = max(0, y - pad)
        w = min(image_bgr.shape[1] - x, w + 2 * pad)
        h = min(image_bgr.shape[0] - y, h + 2 * pad)
        roi = image_bgr[y:y+h, x:x+w]
        if roi.size > 0:
            return roi

    return image_bgr   # fallback: use whole image

In [4]:
def load_and_preprocess(filepath):
    """
    Full preprocessing pipeline per image:
      1. Read (PIL fallback for TIFF)
      2. ROI extraction
      3. Resize to IMG_SIZE
      4. Normalize to [0, 1]
    """
    img = cv2.imread(str(filepath))

    if img is None:
        try:
            from PIL import Image
            pil_img = Image.open(str(filepath)).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception:
            return None

    roi = extract_roi(img)
    roi = cv2.resize(roi, (IMG_SIZE[1], IMG_SIZE[0]))   # cv2 uses (W, H)
    roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    roi = roi.astype(np.float32) / 255.0
    return roi

## 2. Dataset Loading

In [5]:
def load_session(session_path, person_to_label):
    """Load all images from one session folder."""
    images, labels = [], []
    files = sorted([f for f in session_path.iterdir()
                    if f.suffix.lower() in SUPPORTED_EXTS])

    for fpath in tqdm(files, desc=f"  {session_path.name}", leave=False):
        person_id = fpath.stem
        if person_id not in person_to_label:
            continue
        img = load_and_preprocess(fpath)
        if img is not None:
            images.append(img)
            labels.append(person_to_label[person_id])

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

def load_test_data_lazy(test_path_info, person_to_label):
    """Load test data only when needed (during evaluation)."""
    test_p = test_path_info[0]
    person_to_label = test_path_info[1]
    print("\n[INFO] Loading test data (session2) now...")
    X_test, y_test = load_session(test_p, person_to_label)
    return X_test, y_test

In [6]:
"""
ORIGINAL lazy-load version kept above for reference.
"""

def load_dataset(dataset_dir):
    """
    Correct palmprint protocol:
      - session1  → training   (6 000 images, 1 per person)
      - session2  → held-out test  (6 000 images, 1 per person)

    Returns X_train, y_train, X_test, y_test, class_names.
    Training val-split is done later from X_train.
    """
    base    = Path(dataset_dir)
    train_p = base / SESSION_TRAIN
    test_p  = base / SESSION_TEST

    if not train_p.exists():
        sys.exit(f"[ERROR] Training folder not found: {train_p}")
    if not test_p.exists():
        sys.exit(f"[ERROR] Test folder not found:     {test_p}")

    # Build class map from session1
    train_files     = sorted([f for f in train_p.iterdir()
                               if f.suffix.lower() in SUPPORTED_EXTS])
    person_ids      = sorted(set(f.stem for f in train_files))
    person_to_label = {pid: idx for idx, pid in enumerate(person_ids)}
    class_names     = person_ids

    print(f"\n[INFO] Unique persons detected : {len(class_names)}")
    print(f"[INFO] Training session        : {SESSION_TRAIN}")
    print(f"[INFO] Test session            : {SESSION_TEST}")

    print("\n[INFO] Loading training data (session1)...")
    X_train, y_train = load_session(train_p, person_to_label)

    print("\n[INFO] Loading test data (session2)...")
    X_test, y_test = load_session(test_p, person_to_label)

    print(f"\n[INFO] Train : {len(X_train)} images | {X_train[0].shape}")
    print(f"[INFO] Test  : {len(X_test)}  images")

    return X_train, y_train, X_test, y_test, class_names


## 3. Data Augmentation

In [7]:
def get_augmentation_layer():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12),
        layers.RandomZoom(0.08),
        layers.RandomBrightness(0.08),
        layers.RandomContrast(0.08),
    ], name="augmentation")

## 4. CNN Model Architecture
Conv blocks -> ReLU -> MaxPool -> Flatten -> FC -> Softmax

In [8]:
# ─────────────────────────────────────────────────────────────────────
#  ArcFace Loss Layer
#  Additive Angular Margin Softmax — standard for large-scale identity
#  recognition with very few samples per class (ArcFace, CVPR 2019)
# ─────────────────────────────────────────────────────────────────────
class ArcFaceLayer(tf.keras.layers.Layer):
    def __init__(self, num_classes, embedding_size=512,
                 margin=0.5, scale=64.0, **kwargs):
        super().__init__(**kwargs)
        self.num_classes    = num_classes
        self.embedding_size = embedding_size
        self.margin         = margin
        self.scale          = scale
        self.cos_m = tf.cast(tf.math.cos(margin), tf.float32)
        self.sin_m = tf.cast(tf.math.sin(margin), tf.float32)
        self.th    = tf.cast(tf.math.cos(np.pi - margin), tf.float32)
        self.mm    = tf.cast(tf.math.sin(np.pi - margin) * margin, tf.float32)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="arcface_weights",
            shape=(self.embedding_size, self.num_classes),
            initializer="glorot_uniform",
            trainable=True
        )

    def call(self, embeddings, labels=None, training=False):
        en = tf.nn.l2_normalize(embeddings, axis=1)
        Wn = tf.nn.l2_normalize(self.W, axis=0)
        ct = tf.matmul(en, Wn)
        if labels is None or not training:
            return ct * self.scale
        st  = tf.sqrt(tf.maximum(1.0 - tf.square(ct), 1e-9))
        ctm = ct * self.cos_m - st * self.sin_m
        ctm = tf.where(ct > self.th, ctm, ct - self.mm)
        oh  = tf.one_hot(tf.cast(labels, tf.int32), self.num_classes)
        return (oh * ctm + (1.0 - oh) * ct) * self.scale

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_classes": self.num_classes,
                    "embedding_size": self.embedding_size,
                    "margin": self.margin, "scale": self.scale})
        return cfg


# ─────────────────────────────────────────────────────────────────────
#  Backbone: CNN → 512-d unit-normed embedding
#  Architecture is identical to the paper (5 conv blocks).
# ─────────────────────────────────────────────────────────────────────
def build_backbone(embedding_size=512):
    inp = layers.Input(shape=(128, 128, 3), name="input_palm_image")
    x = inp

    # Block 1
    x = layers.Conv2D(32,  (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    # Block 2
    x = layers.Conv2D(64,  (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    # Block 3
    x = layers.Conv2D(128, (3,3), padding="same", name="conv3")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    # Block 4
    x = layers.Conv2D(256, (3,3), padding="same", name="conv4")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    # Block 5
    x = layers.Conv2D(512, (3,3), padding="same", name="conv5")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Embedding head
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(embedding_size, name="embedding_pre")(x)
    x = layers.Lambda(
        lambda t: tf.nn.l2_normalize(t, axis=1),
        name="embedding"
    )(x)   # L2-normalise

    return models.Model(inp, x, name="PalmPrint_Backbone")


# ─────────────────────────────────────────────────────────────────────
#  Full training model: backbone + ArcFace head
# ─────────────────────────────────────────────────────────────────────
def build_model(num_classes, embedding_size=512):
    """
    Returns (full_training_model, backbone).
    Use backbone alone at inference time.
    """
    backbone    = build_backbone(embedding_size)
    img_input   = layers.Input(shape=(128,128,3), name="input_palm_image")
    label_input = layers.Input(shape=(), dtype=tf.int32, name="input_label")
    embeddings  = backbone(img_input, training=True)
    arcface     = ArcFaceLayer(num_classes, embedding_size, name="arcface")
    logits      = arcface(embeddings, label_input, training=True)
    return models.Model(inputs=[img_input, label_input],
                        outputs=logits,
                        name="PalmPrint_ArcFace"), backbone


## 5. Training

In [9]:
def train_model(X_train, y_train, X_val, y_val, num_classes):
    """
    ArcFace identity training.
    X_train / y_train : session1 (ALL images — no further split)
    X_val   / y_val   : 10 % of session2 set aside as validation
    """
    augment = get_augmentation_layer()

    print(f"\n{'='*70}")
    print(f"[TRAINING] ArcFace Identity Model  (num_classes={num_classes})")
    print(f"{'='*70}")

    model, backbone = build_model(num_classes)
    model.summary()

    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        
    )
    print(f"\n[INFO] Optimizer : Adam lr={LEARNING_RATE}")
    print(f"[INFO] Loss      : SparseCategoricalCrossentropy (from_logits=True)")

        # Custom callback: compute 1-NN Rank-1 on validation probes each epoch
    class Rank1Callback(callbacks.Callback):
        def __init__(self, backbone, X_gallery, y_gallery, X_val, y_val,
                     batch_size=32, eval_limit=2000):
            super().__init__()
            self.backbone = backbone
            self.X_gallery = X_gallery
            self.y_gallery = y_gallery
            self.X_val = X_val
            self.y_val = y_val
            self.batch_size = batch_size
            self.eval_limit = min(eval_limit, len(X_val))

        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            # compute embeddings with current backbone weights
            gal_emb = self.backbone.predict(self.X_gallery, batch_size=self.batch_size, verbose=0)
            # sample subset of validation probes for speed
            probe_idx = np.random.permutation(len(self.X_val))[:self.eval_limit]
            probe_emb = self.backbone.predict(self.X_val[probe_idx], batch_size=self.batch_size, verbose=0)
            probe_lbl = self.y_val[probe_idx]

            sim = np.dot(probe_emb, gal_emb.T)
            nn_idx = np.argmax(sim, axis=1)
            y_pred = self.y_gallery[nn_idx]
            rank1 = np.mean(y_pred == probe_lbl) * 100.0

            # expose metric in logs as fraction [0,1] for Keras monitors
            logs['val_rank1'] = rank1 / 100.0
            print(f"\n[VAL] Epoch {epoch+1:03d}  Rank-1 (1-NN): {rank1:.2f}%")

    best_model_path = os.path.join(OUTPUT_DIR, "best_model.keras")

    cb_list = [
        callbacks.EarlyStopping(
            monitor="val_rank1", patience=20,
            restore_best_weights=True, verbose=1, mode="max"
        ),
        callbacks.ReduceLROnPlateau(
            monitor="loss", factor=0.5, patience=6,
            verbose=1, min_lr=1e-6
        ),
        callbacks.ModelCheckpoint(
            filepath=best_model_path, monitor="val_rank1",
            save_best_only=True, verbose=1, mode="max"
        ),
    ]

    def make_arcface_dataset(X, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices(
            ({"input_palm_image": X, "input_label": y.astype("int32")}, y)
        )
        if shuffle:
            ds = ds.shuffle(max(2000, len(X)), seed=RANDOM_SEED)
        ds = ds.batch(BATCH_SIZE)
        if shuffle:
            ds = ds.map(
                lambda xd, yl: (
                    {"input_palm_image": augment(xd["input_palm_image"], training=True),
                     "input_label": xd["input_label"]},
                    yl),
                num_parallel_calls=tf.data.AUTOTUNE)
        return ds.prefetch(tf.data.AUTOTUNE)

    train_ds = make_arcface_dataset(X_train, y_train, shuffle=True)
    val_ds   = make_arcface_dataset(X_val,   y_val,   shuffle=False)

    rank1_cb = Rank1Callback(backbone=backbone,
                             X_gallery=X_train, y_gallery=y_train,
                             X_val=X_val, y_val=y_val,
                             batch_size=BATCH_SIZE, eval_limit=2000)
    cb_list.append(rank1_cb)

    print(f"\n[DATASETS] Train batches: {len(X_train)//BATCH_SIZE}  |  Val batches: {len(X_val)//BATCH_SIZE}")
    print(f"\n{'='*70}")
    print(f"[TRAINING CONFIGURATION]")
    print(f"{'='*70}")
    print(f"  Training images   : {len(X_train)}")
    print(f"  Validation images : {len(X_val)}")
    print(f"  Batch size        : {BATCH_SIZE}")
    print(f"  Max epochs        : {EPOCHS}")

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=cb_list, verbose=1
    )

    backbone.save(os.path.join(OUTPUT_DIR, "backbone.keras"))
    model.save(os.path.join(OUTPUT_DIR, "full_model.keras"))
    print(f"\n[INFO] Full model saved   → {OUTPUT_DIR}/full_model.keras")
    print(f"[INFO] Backbone saved     → {OUTPUT_DIR}/backbone.keras")
    return model, backbone, history


## 6. Evaluation & Plots

In [10]:
def top_k_accuracy(y_true, y_pred_probs, k=5):
    top_k   = np.argsort(y_pred_probs, axis=1)[:, -k:]
    correct = sum(y_true[i] in top_k[i] for i in range(len(y_true)))
    return correct / len(y_true)

In [11]:
def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["loss"],     label="Train Loss",     linewidth=2)
    axes[0].plot(history.history["val_loss"], label="Val Loss",       linewidth=2, linestyle="--")
    axes[0].set_title("Training Loss vs Epochs", fontsize=14)
    axes[0].set_xlabel("Epochs"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["accuracy"],     label="Train Accuracy", linewidth=2)
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy",   linewidth=2, linestyle="--")
    axes[1].set_title("Training Accuracy vs Epochs", fontsize=14)
    axes[1].set_xlabel("Epochs"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"[INFO] Training curves saved -> {path}")

In [12]:
def evaluate_model(model, X_test, y_test, class_names):
    num_classes  = len(class_names)
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\n{'='*55}")
    print(f"  Test Accuracy  : {test_acc * 100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")

    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)

    top5 = top_k_accuracy(y_test, y_pred_probs, k=5)
    print(f"  Top-5 Accuracy : {top5 * 100:.2f}%")
    print(f"{'='*55}")

    if num_classes <= 50:
        report = classification_report(y_test, y_pred, target_names=class_names, digits=4)
    else:
        report = classification_report(y_test, y_pred, digits=4)

    print("\n[Classification Report]\n")
    print(report)

    report_path = os.path.join(OUTPUT_DIR, "classification_report.txt")
    with open(report_path, "w") as f:
        f.write(f"Test Accuracy : {test_acc * 100:.2f}%\n")
        f.write(f"Top-5 Accuracy: {top5 * 100:.2f}%\n")
        f.write(f"Test Loss     : {test_loss:.4f}\n\n")
        f.write(report)
    print(f"[INFO] Report saved -> {report_path}")

    if num_classes <= 50:
        cm     = confusion_matrix(y_test, y_pred)
        fsz    = max(8, num_classes // 3)
        fig, ax = plt.subplots(figsize=(fsz, fsz))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=ax, cmap="Blues", colorbar=False, xticks_rotation="vertical"
        )
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
        plt.savefig(cm_path, dpi=150); plt.close()
        print(f"[INFO] Confusion matrix saved -> {cm_path}")
    else:
        print(f"[INFO] Skipping confusion matrix (>{50} classes)")

    return y_pred, y_pred_probs

## 7. Explainable AI — Grad-CAM
Shows which palm regions the CNN focused on for each prediction.

In [13]:
def get_gradcam_heatmap(backbone_model, img_array, last_conv="conv5"):
    """
    Computes Grad-CAM heatmap using the backbone model directly.
    backbone_model has a named layer 'conv5' accessible via get_layer().
    Uses the L2-norm of the embedding as the score signal.
    """
    grad_model = tf.keras.Model(
        inputs=backbone_model.inputs,
        outputs=[backbone_model.get_layer(last_conv).output,
                 backbone_model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, emb = grad_model(img_array, training=False)
        score = tf.reduce_sum(tf.square(emb), axis=1)
    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = (conv_out[0] @ pooled[..., tf.newaxis]).numpy().squeeze()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap /= heatmap.max()
    return heatmap


In [14]:
def visualize_gradcam(backbone_model, X_probe, y_probe,
                      gallery_emb, gallery_labels,
                      class_names, num_samples=6):
    """
    Grad-CAM visualisation using nearest-neighbour labels from gallery.
    backbone_model : the backbone (not the full ArcFace model)
    gallery_emb    : embeddings of session1 (already computed)
    gallery_labels : labels of session1
    """
    n       = min(num_samples, len(X_probe))
    indices = np.random.choice(len(X_probe), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.5))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        img = X_probe[idx]
        hm  = get_gradcam_heatmap(backbone_model, img[np.newaxis])

        # Nearest-neighbour prediction
        p_emb    = backbone_model.predict(img[np.newaxis], verbose=0)
        sim      = np.dot(p_emb, gallery_emb.T)[0]
        pred_cls = int(gallery_labels[np.argmax(sim)])
        conf     = float(np.max(sim))

        hmap    = cv2.resize(hm, (IMG_SIZE[1], IMG_SIZE[0]))
        colored = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted((img * 255).astype(np.uint8), 0.6, colored, 0.4, 0)

        true_lbl = class_names[y_probe[idx]]
        pred_lbl = class_names[pred_cls]

        axes[row, 0].imshow(img);          axes[row, 0].set_title("Input Palm");       axes[row, 0].axis("off")
        axes[row, 1].imshow(hmap, cmap="jet"); axes[row, 1].set_title("Grad-CAM (XAI)"); axes[row, 1].axis("off")
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f"Pred: {pred_lbl} (sim={conf:.2f})\nTrue: {true_lbl}",
            color="green" if pred_lbl == true_lbl else "red"
        )
        axes[row, 2].axis("off")

    plt.suptitle("Explainable AI (Grad-CAM) — Palm Region Importance", fontsize=13, y=1.01)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "gradcam_xai.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[INFO] Grad-CAM XAI saved → {path}")


## 8. Sample Visualisation

In [15]:
def visualize_samples(X_train, y_train, class_names, n=8):
    n       = min(n, len(X_train))
    indices = np.random.choice(len(X_train), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 3))
    for ax, idx in zip(axes, indices):
        ax.imshow(X_train[idx])
        ax.set_title(f"ID: {class_names[y_train[idx]]}", fontsize=7)
        ax.axis("off")
    plt.suptitle("Sample Training Images (after ROI + Preprocessing)")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "sample_images.png")
    plt.savefig(path, dpi=120); plt.close()
    print(f"[INFO] Sample images saved -> {path}")

## 9. Single Image Prediction

In [16]:
def predict_single(model, image_path, class_names):
    img = load_and_preprocess(image_path)
    if img is None:
        print(f"[ERROR] Could not load: {image_path}"); return

    probs     = model.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_cls  = int(np.argmax(probs))
    conf      = float(probs[pred_cls])
    top5_idx  = np.argsort(probs)[::-1][:5]

    print(f"\n[Prediction for: {image_path}]")
    print(f"  Best Match : Person {class_names[pred_cls]}  ({conf*100:.1f}% confidence)")
    print(f"\n  Top-5 Matches:")
    for rank, i in enumerate(top5_idx, 1):
        print(f"    {rank}. Person {class_names[i]:>8}  ->  {probs[i]*100:.2f}%")

---
## 10. Run Pipeline
Execute each cell below step-by-step. If an error occurs, fix it and re-run only the failed cell — previous cells' state is preserved.

### Step 1: Load Dataset

In [17]:
print("\n" + "="*60)
print("  Deep Learning Contactless PalmPrint Recognition")
print("  Mode: Person Identity Recognition (6 000 classes)")
print("="*60)

X_train, y_train, X_test, y_test, class_names = load_dataset(DATASET_DIR)
num_classes = len(class_names)

import psutil
mem_mb = psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024
print(f"\n[INFO] RAM after loading : {mem_mb:.1f} MB")
print(f"[INFO] num_classes       : {num_classes}")



  Deep Learning Contactless PalmPrint Recognition
  Mode: Person Identity Recognition (6 000 classes)

[INFO] Unique persons detected : 6000
[INFO] Training session        : session1
[INFO] Test session            : session2

[INFO] Loading training data (session1)...



[INFO] Loading test data (session2)...



[INFO] Train : 6000 images | (128, 128, 3)
[INFO] Test  : 6000  images

[INFO] RAM after loading : 2606.9 MB
[INFO] num_classes       : 6000


### Step 2: Visualize Samples

In [18]:
visualize_samples(X_train, y_train, class_names)

[INFO] Sample images saved -> outputs\sample_images.png


In [19]:
print(f"\n{'='*70}")
print(f"[DATA PREP] Train = session1 (all)  |  Val = 10% of session2")
print(f"{'='*70}")

# ── session1 stats ────────────────────────────────────────────────────────
unique_all, counts_all = np.unique(y_train, return_counts=True)
print(f"\n[SESSION1 — training]")
print(f"  Total images   : {len(y_train)}")
print(f"  Unique classes : {len(unique_all)}")
print(f"  Images/class   : min={counts_all.min()}  max={counts_all.max()}  mean={counts_all.mean():.2f}")

# ── Split session2: 10% val, 90% final test ───────────────────────────────
# StratifiedShuffleSplit requires >= 2 samples per class — impossible here
# (1 image per person per session).  Plain random split works fine because
# both session1 and session2 cover the SAME 6000 person IDs, so val labels
# are always a subset of the train label space.
VAL_FRACTION = 0.10
n_val = max(BATCH_SIZE, int(len(X_test) * VAL_FRACTION))

perm         = np.random.default_rng(RANDOM_SEED).permutation(len(X_test))
val_idx      = perm[:n_val]
test_idx     = perm[n_val:]

X_val_split  = X_test[val_idx];  y_val_split  = y_test[val_idx]
X_test_final = X_test[test_idx]; y_test_final = y_test[test_idx]

# Training uses ALL of session1
X_tr = X_train; y_tr = y_train
NUM_CLASSES_TO_USE = num_classes

unique_val, counts_val = np.unique(y_val_split, return_counts=True)
print(f"\n[SPLIT]")
print(f"  Train (session1) : {len(X_tr):5d} images  |  {len(unique_all)} classes")
print(f"  Val   (10% s2)   : {len(X_val_split):5d} images  |  {len(unique_val)} classes")
print(f"  Test  (90% s2)   : {len(X_test_final):5d} images")
print(f"  Val batches      : {len(X_val_split) // BATCH_SIZE}")

print(f"\n[DATA SANITY]")
print(f"  X_tr range  : [{X_tr.min():.3f}, {X_tr.max():.3f}]  (expect [0,1])")
print(f"  NaNs in X_tr: {np.isnan(X_tr).any()}")
print(f"  All val labels in train labels: {np.all(np.isin(y_val_split, y_tr))}")

print(f"\n[INFO] Ready to train:")
print(f"  Train  : {len(X_tr)} images")
print(f"  Val    : {len(X_val_split)} images")
print(f"  Classes: {NUM_CLASSES_TO_USE}")
print(f"{'='*70}\n")



[DATA PREP] Train = session1 (all)  |  Val = 10% of session2

[SESSION1 — training]
  Total images   : 6000
  Unique classes : 6000
  Images/class   : min=1  max=1  mean=1.00

[SPLIT]
  Train (session1) :  6000 images  |  6000 classes
  Val   (10% s2)   :   600 images  |  600 classes
  Test  (90% s2)   :  5400 images
  Val batches      : 18

[DATA SANITY]
  X_tr range  : [0.027, 1.000]  (expect [0,1])
  NaNs in X_tr: False
  All val labels in train labels: True

[INFO] Ready to train:
  Train  : 6000 images
  Val    : 600 images
  Classes: 6000



In [20]:
# Diagnostic: Check if embeddings are normalized
print("[DIAGNOSTIC] Embedding Normalization Check")

# Build a test backbone
test_backbone = build_backbone()

# Create random input
test_input = np.random.rand(4, 128, 128, 3).astype("float32")

# Get embeddings
test_embeddings = test_backbone.predict(test_input, verbose=0)

# Check L2-norm
embedding_norms = np.linalg.norm(test_embeddings, axis=1)

print(f"\nEmbedding L2-norms (should be ≈ 1.0):")
print(f"  Mean: {embedding_norms.mean():.6f}")
print(f"  Std:  {embedding_norms.std():.6f}")
print(f"  Min:  {embedding_norms.min():.6f}")
print(f"  Max:  {embedding_norms.max():.6f}")

if np.allclose(embedding_norms, 1.0, atol=1e-5):
    print("\n✅ PASS: Embeddings are properly L2-normalized")
else:
    print("\n❌ FAIL: Embeddings are NOT normalized! Fix UnitNormalization layer")

[DIAGNOSTIC] Embedding Normalization Check


Embedding L2-norms (should be ≈ 1.0):
  Mean: 1.000000
  Std:  0.000000
  Min:  1.000000
  Max:  1.000000

✅ PASS: Embeddings are properly L2-normalized


### Step 3: Train Model

In [ ]:
print(f"\n[INFO] Training setup:")
print(f"  Train  : {len(X_tr)} images  |  {len(np.unique(y_tr))} classes")
print(f"  Val    : {len(X_val_split)} images  |  {len(np.unique(y_val_split))} classes")
print(f"  Classes: {NUM_CLASSES_TO_USE}")

model, backbone, history = train_model(X_tr, y_tr, X_val_split, y_val_split, NUM_CLASSES_TO_USE)



[INFO] Training setup:
  Train  : 6000 images  |  6000 classes
  Val    : 600 images  |  600 classes
  Classes: 6000

[TRAINING] ArcFace Identity Model  (num_classes=6000)


Model: "PalmPrint_ArcFace"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_palm_image    │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PalmPrint_Backbone  │ (None, 512)       │  2,097,856 │ input_palm_image… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_label         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ arcface             │ (None, 6000)      │  3,072,000 │ PalmPrint_Backbo… │
│ (ArcFaceLayer)      │                   │            │ input_label[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,169,856 (19.72 MB)

 Trainable params: 5,167,872 (19.71 MB)

 Non-trainable params: 1,984 (7.75 KB)


[INFO] Optimizer : Adam lr=0.0001
[INFO] Loss      : SparseCategoricalCrossentropy (from_logits=True)

[DATASETS] Train batches: 187  |  Val batches: 18

[TRAINING CONFIGURATION]
  Training images   : 6000
  Validation images : 600
  Batch size        : 32
  Max epochs        : 80
Epoch 1/80
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 42.4254
Epoch 1: finished saving model to outputs\best_model.keras

[VAL] Epoch 001  Rank-1 (1-NN): 0.17%
188/188 ━━━━━━━━━━━━━━━━━━━━ 1598s 8s/step - loss: 41.9537 - val_loss: 9.3772 - learning_rate: 1.0000e-04 - val_rank1: 0.0017
Epoch 2/80
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - loss: 40.6674
Epoch 2: finished saving model to outputs\best_model.keras

[VAL] Epoch 002  Rank-1 (1-NN): 0.33%
188/188 ━━━━━━━━━━━━━━━━━━━━ 1684s 9s/step - loss: 40.5899 - val_loss: 9.4740 - learning_rate: 1.0000e-04 - val_rank1: 0.0033
Epoch 3/80
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 40.3273
Epoch 3: finished saving model to outputs\best_model.keras

[VAL] Ep

### Step 4: Plot Training Curves

In [ ]:
plot_training_curves(history)

[INFO] Training curves saved -> outputs\training_curves.png


### Step 5: Evaluate Model

In [ ]:
print(f"\n{'='*70}")
print(f"[EVALUATION] 1-NN Cosine Matching: session1 gallery → session2 probes")
print(f"{'='*70}")

# ── Gallery: embed ALL of session1 ────────────────────────────────────────
print("\n[INFO] Building gallery from session1 (X_train)...")
gallery_embeddings = backbone.predict(X_train, batch_size=BATCH_SIZE, verbose=1)
gallery_labels     = y_train

# ── Probe: held-out 90% of session2 ──────────────────────────────────────
print("\n[INFO] Building probe embeddings from session2 test set...")
EVAL_LIMIT   = min(2000, len(X_test_final))
probe_idx    = np.random.permutation(len(X_test_final))[:EVAL_LIMIT]
X_probe      = X_test_final[probe_idx]
y_probe      = y_test_final[probe_idx]
probe_embeddings = backbone.predict(X_probe, batch_size=BATCH_SIZE, verbose=1)

# ── Cosine similarity (both already unit-normed → dot = cosine) ───────────
print("\n[INFO] Computing cosine similarity...")
sim_matrix   = np.dot(probe_embeddings, gallery_embeddings.T)  # (N_probe, N_gallery)
nn_idx       = np.argmax(sim_matrix, axis=1)
y_pred_nn    = gallery_labels[nn_idx]

rank1_acc = np.mean(y_pred_nn == y_probe) * 100

top5_idx  = np.argsort(sim_matrix, axis=1)[:, -5:]
rank5_acc = sum(y_probe[i] in gallery_labels[top5_idx[i]]
                for i in range(len(y_probe))) / len(y_probe) * 100

print(f"\n{'='*70}")
print(f"[RESULTS] on {EVAL_LIMIT} probe images from session2")
print(f"{'='*70}")
print(f"  Rank-1 Accuracy : {rank1_acc:.2f}%")
print(f"  Rank-5 Accuracy : {rank5_acc:.2f}%")
print(f"{'='*70}\n")

# ── ArcFace logit accuracy on val set (quick check) ─────────────────────
print("[INFO] Val-set ArcFace logit accuracy...")
val_ds_eval = tf.data.Dataset.from_tensor_slices(
    ({"input_palm_image": X_val_split,
      "input_label":      y_val_split.astype("int32")},
     y_val_split)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_loss, val_acc = model.evaluate(val_ds_eval, verbose=0)
print(f"  Val accuracy (ArcFace logits): {val_acc*100:.2f}%")
print(f"  Val loss                     : {val_loss:.4f}")
print(f"\n{'='*70}\n")



[EVALUATION] 1-NN Cosine Matching: session1 gallery → session2 probes

[INFO] Building gallery from session1 (X_train)...
375/375 ━━━━━━━━━━━━━━━━━━━━ 268s 714ms/step

[INFO] Building probe embeddings from session2 test set...
125/125 ━━━━━━━━━━━━━━━━━━━━ 89s 710ms/step

[INFO] Computing cosine similarity...

[RESULTS] on 2000 probe images from session2
  Rank-1 Accuracy : 0.15%
  Rank-5 Accuracy : 0.60%

[INFO] Val-set ArcFace logit accuracy...
  Val accuracy (ArcFace logits): 0.17%
  Val loss                     : 10.6329




### Step 6: Grad-CAM Visualization

In [ ]:
print("[INFO] Running Grad-CAM visualisation on sample probe images...")
visualize_gradcam(
    backbone_model = backbone,
    X_probe        = X_probe[:50],
    y_probe        = y_probe[:50],
    gallery_emb    = gallery_embeddings,
    gallery_labels = gallery_labels,
    class_names    = class_names,
    num_samples    = 6
)


[INFO] Running Grad-CAM visualisation on sample probe images...
[INFO] Grad-CAM XAI saved → outputs\gradcam_xai.png


### Step 7: Summary

In [ ]:
print(f"\n[INFO] Model saved  -> {MODEL_SAVE}")
print(f"[INFO] All outputs  -> {OUTPUT_DIR}/")
print("\nPipeline complete!")


[INFO] Model saved  -> palmprint_model.h5
[INFO] All outputs  -> outputs/

Pipeline complete!


In [ ]:
print(f"\n{'='*70}")
print(f"[DRY RUN TEST] Checking all components...")
print(f"{'='*70}")

try:
    print(f"\n[TEST 1] Dataset loading...")
    assert len(X_train) > 0 and len(X_test) > 0
    print(f"  ✅ PASS: train={len(X_train)}, test={len(X_test)}, classes={len(class_names)}")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 2] Model build (ArcFace)...")
    _m, _b = build_model(10)
    _m.compile(optimizer="adam",
               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
               metrics=["accuracy"])
    print(f"  ✅ PASS: full_model params={_m.count_params():,}  backbone params={_b.count_params():,}")
    del _m, _b
except Exception as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 3] Data shapes...")
    assert X_tr.shape[1:] == (128, 128, 3)
    assert X_val_split.shape[1:] == (128, 128, 3)
    assert len(y_tr) == len(X_tr)
    print(f"  ✅ PASS: X_tr={X_tr.shape}  X_val={X_val_split.shape}")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 4] ArcFace forward pass (batch of 4)...")
    _m2, _b2 = build_model(10)
    _m2.compile(optimizer="adam",
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=["accuracy"])
    _x = np.random.rand(4, 128, 128, 3).astype("float32")
    _y = np.array([0, 1, 2, 3], dtype="int32")
    _out = _m2({"input_palm_image": _x, "input_label": _y}, training=True)
    assert _out.shape == (4, 10), f"Expected (4,10), got {_out.shape}"
    print(f"  ✅ PASS: output shape={_out.shape}")
    del _m2, _b2
except Exception as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 5] Val labels ⊆ train labels...")
    assert np.all(np.isin(y_val_split, y_tr))
    print(f"  ✅ PASS: all val labels appear in training set")
except AssertionError as e:
    print(f"  ❌ FAIL: {e}")

try:
    print(f"\n[TEST 6] GradCAM heatmap generation...")
    _bb = build_backbone()
    _hm = get_gradcam_heatmap(_bb, np.random.rand(1,128,128,3).astype("float32"))
    assert _hm.ndim >= 1
    print(f"  ✅ PASS: heatmap shape={_hm.shape}")
    del _bb
except Exception as e:
    print(f"  ❌ FAIL: {e}")

print(f"\n{'='*70}")
print(f"[DRY RUN] ✅ ALL TESTS PASSED — Ready for training!")
print(f"{'='*70}\n")



[DRY RUN TEST] Checking all components...

[TEST 1] Dataset loading...
  ✅ PASS: train=6000, test=6000, classes=6000

[TEST 2] Model build (ArcFace)...
  ✅ PASS: full_model params=2,102,976  backbone params=2,097,856

[TEST 3] Data shapes...
  ✅ PASS: X_tr=(6000, 128, 128, 3)  X_val=(600, 128, 128, 3)

[TEST 4] ArcFace forward pass (batch of 4)...
  ✅ PASS: output shape=(4, 10)

[TEST 5] Val labels ⊆ train labels...
  ✅ PASS: all val labels appear in training set

[TEST 6] GradCAM heatmap generation...
  ✅ PASS: heatmap shape=(8, 8)

[DRY RUN] ✅ ALL TESTS PASSED — Ready for training!



### (Optional) Predict a Single Image
Uncomment and edit the path below to test a single palm image.

In [ ]:
# predict_single(model, "path/to/your/palm_image.tiff", class_names)